# Evaluating a RAG solution with Giskard

In the final part of this notebook, you will find the necessary code to evaluate the suitability of the responses provided by the Agent using Giskard.

## Installing libraries & Loading Dataset

We will download the dataset from the Hugging Face datasets library. It's a dataset with information about diseases.

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY') 

In [2]:
from datasets import load_dataset

data = load_dataset("keivalya/MedQuad-MedicalQnADataset", split='train')


In [3]:
data = data.to_pandas()
data.head(10)

,qtype,Question,Answer
0,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,symptoms,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
2,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,Individuals of all ages who come into contact ...
3,exams and tests,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos..."
4,treatment,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen..."
5,prevention,How to prevent Lymphocytic Choriomeningitis (L...,LCMV infection can be prevented by avoiding co...
6,information,What is (are) Parasites - Cysticercosis ?,Cysticercosis is an infection caused by the la...
7,susceptibility,Who is at risk for Parasites - Cysticercosis? ?,Cysticercosis is an infection caused by the la...
8,exams and tests,How to diagnose Parasites - Cysticercosis ?,"If you think that you may have cysticercosis, ..."
9,treatment,What are the treatments for Parasites - Cystic...,Some people with cysticercosis do not need to ...


In [4]:
data = data[0:100]

As you can see, the medical information in the dataset is well-organized, and to someone like me, who is not an expert in the field, it appears to be quite valuable. This information could be a useful addition to any general medicine book to support primary care doctors.

Load the langchain libraries to load the document.

In [5]:
from langchain.document_loaders import DataFrameLoader
from langchain.vectorstores import Chroma

The Document is in the Answer column, and the others columns are Metadata.

In [6]:
df_loader = DataFrameLoader(data, page_content_column="Answer")


In [7]:
df_document = df_loader.load()
display(df_document[:2])

[Document(metadata={'qtype': 'susceptibility', 'Question': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?'}, page_content='LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.'),
 Document(metadata={'qtype': 'symptoms', 'Question': 'What are the symptoms of Lymphocytic Choriomeningitis (LCM) ?'}, page_content='LCMV is most commonly recognized as causing neurological disease, as its name implies, though infection without symptoms or mild febrile illnesses are more common clinical manifestations. \n                \nFor infected persons who do become ill, onset of sympt

We can chunk the documents. The size to which we want to split the document is a design decision. The larger it is, the larger the prompt will be, and the slower the Model's response process.

We also need to consider the maximum prompt size and ensure that the document does not exceed it.

In [8]:
from langchain.text_splitter import CharacterTextSplitter

In [9]:
text_splitter = CharacterTextSplitter(chunk_size=1250, chunk_overlap=100)
texts = text_splitter.split_documents(df_document)


These warnings we see are because it can't perform the partition of the required size. This is because it waits for a page break to divide the text and does so when possible.

In [10]:
first_doc = texts[1]
print(first_doc.page_content)

LCMV is most commonly recognized as causing neurological disease, as its name implies, though infection without symptoms or mild febrile illnesses are more common clinical manifestations. 
                
For infected persons who do become ill, onset of symptoms usually occurs 8-13 days after exposure to the virus as part of a biphasic febrile illness. This initial phase, which may last as long as a week, typically begins with any or all of the following symptoms: fever, malaise, lack of appetite, muscle aches, headache, nausea, and vomiting. Other symptoms appearing less frequently include sore throat, cough, joint pain, chest pain, testicular pain, and parotid (salivary gland) pain. 
                
Following a few days of recovery, a second phase of illness may occur. Symptoms may consist of meningitis (fever, headache, stiff neck, etc.), encephalitis (drowsiness, confusion, sensory disturbances, and/or motor abnormalities, such as paralysis), or meningoencephalitis (inflammation 

### Initialize the Embedding Model and Vector DB

We load the text-embedding-ada-002 model from OpenAI.

In [11]:
from langchain_openai import OpenAIEmbeddings

model_name = 'text-embedding-ada-002'
#model_name = 'text-embedding-3-small'

embed = OpenAIEmbeddings(
    model=model_name,
    openai_api_key=OPENAI_API_KEY
)

The execution of this cell may take 3 to 5 minutes. If you want it to be faster, you can reduce the number of records in the dataset.

In [12]:
directory_cdb = 'chromadb/'
chroma_db = Chroma.from_documents(
    df_document, embed, persist_directory=directory_cdb
)

We are going to create three objects.

* The language model, which can be any of those from OpenAI, the most common being gpt-3.5.
* The memory, responsible for keeping the prompt with all the necessary history.
* The retrieval, used to obtain information stored in ChromaDB.

In [37]:
from langchain.chat_models import ChatOpenAI
from langchain_openai import OpenAI
from langchain.chains.conversation.memory import ConversationBufferWindowMemory
from langchain.chains import RetrievalQA

llm=OpenAI(openai_api_key=OPENAI_API_KEY,
           temperature=0.0, model_name="gpt-3.5-turbo-instruct")

conversational_memory = ConversationBufferWindowMemory(
    memory_key='chat_history',
    k=4, #Number of messages stored in memory
    return_messages=True #Must return the messages in the response.
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=chroma_db.as_retriever()
)

We can try the isolated Retrieval to see if the information it returns is relevant.




In [38]:
qa.invoke("What is the main symptom of LCM?")

{'query': 'What is the main symptom of LCM?',
 'result': ' The main symptom of LCM is a biphasic febrile illness, which includes symptoms such as fever, malaise, lack of appetite, muscle aches, headache, nausea, and vomiting.'}

## Creating the Agent.

In [39]:
from langchain.agents import Tool, AgentExecutor

#Defining the list of tool objects to be used by LangChain.
tools = [
    Tool(
        name='Medical KB',
        func=qa.run,
        description=(
            'use this tool when answering medical knowledge queries to get '
            'more information about the topic'
        )
    )
]

In [40]:
from langchain.agents import initialize_agent, create_react_agent
from langchain import hub

prompt = hub.pull("hwchase17/react-chat")
agent = create_react_agent(
    #agent='chat-conversational-react-description',
    tools=tools,
    llm=llm,
    prompt=prompt
    #verbose=True,
    #max_iterations=3,
    #early_stopping_method='generate',
    #memory=conversational_memory
)

In [41]:
# Create an agent executor by passing in the agent and tools
agent_executor = AgentExecutor(agent=agent,
                               tools=tools,
                               verbose=True,
                               memory=conversational_memory,
                               max_iterations=30,
                               max_execution_time=600,
                               #early_stopping_method='generate',
                               handle_parsing_errors=True
                               )

### Using the Conversational Agent

To make queries we simply call the `agent` directly.

First i will try a order not related to the Medical field.

In [42]:
agent_executor.invoke({"input": "Give me the area of square of 2x2"})



> Entering new AgentExecutor chain...

Thought: Do I need to use a tool? Yes
Action: Medical KB
Action Input: Area of square I don't know.Do I need to use a tool? No
Final Answer: The area of a square with sides of 2 units is 4 square units.

> Finished chain.


{'input': 'Give me the area of square of 2x2',
 'chat_history': [],
 'output': 'The area of a square with sides of 2 units is 4 square units.'}

Perfect, the model has responded without accessing the configured knowledge database.

Now I will try with a question that is also not related to health.

In [19]:
agent_executor.invoke({"input": "Do you know who is Clark Kent?"})



> Entering new AgentExecutor chain...

Thought: Do I need to use a tool? Yes
Action: Medical KB
Action Input: Clark Kent I don't know.Do I need to use a tool? No
Final Answer: Clark Kent is the secret identity of Superman, a superhero in DC Comics. He is a journalist for the Daily Planet and lives in Metropolis.

> Finished chain.


{'input': 'Do you know who is Clark Kent?',
 'chat_history': [HumanMessage(content='Give me the area of square of 2x2', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The area of a square with sides of 2 units is 4 square units.', additional_kwargs={}, response_metadata={})],
 'output': 'Clark Kent is the secret identity of Superman, a superhero in DC Comics. He is a journalist for the Daily Planet and lives in Metropolis.'}

It has not accessed either, as the model has been able to identify that it is not a question related to the database that LangChain provides.

Now it's time to try with a question related to Medicine. Let's see if the model can understand that it should first look for information in the vector database at its disposal.

In [43]:
 agent_executor.memory.clear()

In [44]:
agent_executor.invoke({"input": """I have a patient that can have Botulism,
how can I confirm the diagnose?"""})



> Entering new AgentExecutor chain...

Thought: Do I need to use a tool? Yes
Action: Medical KB
Action Input: Botulism Botulism is a rare but serious paralytic illness caused by a nerve toxin produced by certain bacteria. It can be contracted through contaminated food, wounds, or ingestion of bacterial spores. Symptoms include muscle paralysis, difficulty swallowing, and respiratory failure. Treatment includes antitoxin, supportive care, and removal of contaminated food or wound.Do I need to use a tool? No
Final Answer: To confirm a diagnosis of botulism, a doctor may perform a physical exam, review symptoms and medical history, and order laboratory tests such as a stool or blood test. They may also ask about recent food consumption and potential exposure to contaminated food or wounds. It is important to seek medical attention immediately if botulism is suspected.

> Finished chain.


{'input': 'I have a patient that can have Botulism,\nhow can I confirm the diagnose?',
 'chat_history': [],
 'output': 'To confirm a diagnosis of botulism, a doctor may perform a physical exam, review symptoms and medical history, and order laboratory tests such as a stool or blood test. They may also ask about recent food consumption and potential exposure to contaminated food or wounds. It is important to seek medical attention immediately if botulism is suspected.'}

Perfect, the most important thing for us is that it has been able to identify that it should go to the medical database to search for information about the symptoms.

In [45]:
agent_executor.invoke({"input": "Is this an important illness?"})



> Entering new AgentExecutor chain...

Thought: Do I need to use a tool? No
Final Answer: Yes, botulism is a serious illness that requires immediate medical attention. It is caused by a toxin produced by the bacteria Clostridium botulinum and can lead to paralysis and even death if left untreated. If you suspect you or someone you know may have botulism, it is important to seek medical help right away.

> Finished chain.


{'input': 'Is this an important illness?',
 'chat_history': [HumanMessage(content='I have a patient that can have Botulism,\nhow can I confirm the diagnose?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='To confirm a diagnosis of botulism, a doctor may perform a physical exam, review symptoms and medical history, and order laboratory tests such as a stool or blood test. They may also ask about recent food consumption and potential exposure to contaminated food or wounds. It is important to seek medical attention immediately if botulism is suspected.', additional_kwargs={}, response_metadata={})],
 'output': 'Yes, botulism is a serious illness that requires immediate medical attention. It is caused by a toxin produced by the bacteria Clostridium botulinum and can lead to paralysis and even death if left untreated. If you suspect you or someone you know may have botulism, it is important to seek medical help right away.'}

And the memory works perfectly. We can maintain a conversation, taking into account that the model knows the previous questions and answers.

## Evaluating the solution with Giskard

Install and Load the libraries.

In [24]:
#!pip install --default-timeout=100 "giskard[llm]"
from giskard.rag import KnowledgeBase, generate_testset, evaluate

Is necesary to create a Dataframe with just the column containing the information used to create the RAG system.

In [25]:
import pandas as pd
df_giskard = pd.DataFrame([d.page_content for d in df_document], columns=["text"])
df_giskard.head(10)

,text
0,LCMV infections can occur after exposure to fr...
1,LCMV is most commonly recognized as causing ne...
2,Individuals of all ages who come into contact ...
3,"During the first phase of the disease, the mos..."
4,"Aseptic meningitis, encephalitis, or meningoen..."
5,LCMV infection can be prevented by avoiding co...
6,Cysticercosis is an infection caused by the la...
7,Cysticercosis is an infection caused by the la...
8,"If you think that you may have cysticercosis, ..."
9,Some people with cysticercosis do not need to ...


Using the information from the dataset, we ask Giskard to create a Knowledge Base, which is nothing more than a set of questions along with their respective answers. Both the questions and answers are generated by OpenAI's most advanced model, which is why it requires our OpenAI key to be provided.

In [26]:
kb_giskard = KnowledgeBase(df_giskard)

In [29]:
# The more questions you generate, the more you will be charged.
test_questions = generate_testset(
    kb_giskard,
    num_questions=30,
    agent_description="Medical assistant for diagnosis and treatment support.",
)

2024-10-26 17:54:27,725 pid:88127 MainThread giskard.rag  INFO     Finding topics in the knowledge base.
2024-10-26 17:54:35,893 pid:88127 MainThread giskard.rag  INFO     Found 3 topics in the knowledge base.


Generating questions:   0%|          | 0/30 [00:00<?, ?it/s]

In [30]:
df_test_questions = test_questions.to_pandas()

In [31]:
df_test_questions.head()

,question,reference_answer,reference_context,conversation_history,metadata
id,,,,,
d4474e30-5c9c-4e24-8f3f-1690883847a4,What laboratory tests can be used to confirm a...,Antigen-capture enzyme-linked immunosorbent as...,Document 90: Many of the signs and symptoms of...,[],"{'question_type': 'simple', 'seed_document_id'..."
652cf1a6-1dbc-4970-86c9-a102839a7b9b,What is the primary risk factor for acquiring ...,Eating raw or undercooked beef or pork is the ...,Document 7: Cysticercosis is an infection caus...,[],"{'question_type': 'simple', 'seed_document_id'..."
f52dc599-b29c-4b8e-a497-5df613884138,What is the role of the Centers for Disease Co...,The Centers for Disease Control and Prevention...,Document 21: Some health departments test shel...,[],"{'question_type': 'simple', 'seed_document_id'..."
3103b048-3cec-441f-a45c-ea4198d9ec9f,What is the current treatment for Marburg hemo...,There is no specific treatment for Marburg hem...,Document 91: There is no specific treatment fo...,[],"{'question_type': 'simple', 'seed_document_id'..."
caba2fac-cee9-43ea-8ee6-67950b0aea91,What methods are used to diagnose toxocariasis?,Toxocariasis can be diagnosed through a blood ...,Document 42: If you think you or your child ma...,[],"{'question_type': 'simple', 'seed_document_id'..."


A function is created that will be called from Giskard's evaluate function. This function receives the question and returns the agent's response.

In [ ]:
# df_test_questions['reference_context'][0]

In [58]:
def use_agent(question, history=None):
    response = agent_executor.invoke({"input": question})
    return response["output"]

In [59]:
report = evaluate(use_agent, testset=test_questions, knowledge_base=kb_giskard)

Asking questions to the agent:   0%|          | 0/30 [00:00<?, ?it/s]



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Medical KB
Action Input: Marburg Hemorrhagic Fever Marburg hemorrhagic fever is a rare and potentially deadly disease caused by the Marburg virus. It is difficult to diagnose because its symptoms are similar to other more common infectious diseases. There is no specific treatment for Marburg hemorrhagic fever, but supportive hospital therapy can help manage symptoms. Preventive measures include avoiding contact with fruit bats and sick non-human primates, and using barrier nursing techniques to prevent person-to-person transmission. Increasing awareness and improving diagnostic tools are important in controlling the spread of the disease. The case-fatality rate for Marburg hemorrhagic fever can range from 23-90%.Do I need to use a tool? No
Final Answer: Laboratory tests such as blood tests, urine tests, and tissue samples can be used to confirm a case of Marburg Hemorrhagic Fever.

> Finished chain.



CorrectnessMetric evaluation:   0%|          | 0/30 [00:00<?, ?it/s]

In [60]:
# Summary with the results.
report.correctness_by_question_type()

,correctness
question_type,
complex,0.6
conversational,0.2
distracting element,0.8
double,0.4
simple,0.8
situational,0.4


In [61]:
# Obtaining the incorrect answers
failures = report.get_failures()[:2]
failures

,question,reference_answer,reference_context,conversation_history,metadata,agent_answer,correctness,correctness_reason
id,,,,,,,,
d4474e30-5c9c-4e24-8f3f-1690883847a4,What laboratory tests can be used to confirm a...,Antigen-capture enzyme-linked immunosorbent as...,Document 90: Many of the signs and symptoms of...,[],"{'question_type': 'simple', 'seed_document_id'...","Laboratory tests such as blood tests, urine te...",False,The agent provided a generic answer about labo...
927d69f6-8094-4b55-aaef-1507ae0b9812,"Can you explain the origin of Cysticercosis, t...",Cysticercosis is an infection caused by the la...,Document 6: Cysticercosis is an infection caus...,[],"{'question_type': 'complex', 'seed_document_id...",Cysticercosis is caused by the larvae of the t...,False,The agent's answer was partially correct but l...


In [62]:
# Giskard explains the reasons why it considers the answers to be incorrect.
failures['correctness_reason'].iloc[1]

"The agent's answer was partially correct but lacked detail. The agent stated that Cysticercosis is contracted through contaminated food or water, but should have specified that it is contracted when a person swallows eggs excreted in the stool of people with the adult tapeworm. The agent also stated that to avoid contracting it, one should avoid eating raw or undercooked pork, but should have added that maintaining good hygiene is also important."


# Conclusions.
The experiment has been a small success. The Vectorial database has been configured and filled with information from the dataset. A LangChain agent has been created, and it has been able to retrieve information from the database only when necessary. Don't forget that our ChatBot has memory.

All of this in just a few lines of code!

You should do a deeper dive with the [Giskard official tutorials](https://docs.giskard.ai/en/stable/open_source/testset_generation/testset_generation/index.html)


---